# End-to-end surrogate workflow: AL → GP → BO

**Problem framing.** You have an expensive black-box `f(L) = VSWR @ 1 GHz of a thin monopole of length L`. Each query costs (say) 30 seconds of FDTD time. You want the optimum length quickly.

**Strategy.**

- **Phase 1 — Active learning (AL).** Use variance-acquisition AL to spread 8 samples across the design space, so the GP surrogate has good coverage everywhere before we commit to a region.
- **Phase 2 — GP retrain with ML hyperparameters.** Fit a fresh `GaussianProcess` on the AL samples, maximizing the log marginal likelihood so the lengthscale / signal variance / noise are not hand-tuned.
- **Phase 3 — Bayesian optimization (BO).** Run BO with Expected Improvement to drill into the optimum. Compare the convergence to the AL-bootstrapped GP.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
from yee import AlConfig, active_learn
from yee import GaussianProcess
from yee import BoConfig, bo_minimize

Synthetic objective: VSWR for a monopole near λ/4 at 1 GHz. Closed-form so the notebook runs in seconds; the API is identical to wrapping a real `FdtdDriver` call.

In [ ]:
C_LIGHT = 2.99792458e8
F0 = 1.0e9
LAMBDA_QUARTER = C_LIGHT / F0 / 4.0   # 0.07495 m

def vswr(x):
    length = float(x[0])
    offset = (length - LAMBDA_QUARTER) / LAMBDA_QUARTER
    return 1.0 + 50.0 * offset**2 + 0.3 * math.sin(20.0 * length)

## Phase 1 — Active learning bootstrap

Spread 8 AL samples across `L ∈ [0.04, 0.12] m`. The variance acquisition will pick points where the GP is least certain — wide initial coverage rather than chasing the optimum.

In [ ]:
al_cfg = AlConfig(n_initial=3, n_iters=5, n_candidates=1024, seed=42)
bounds = [(0.04, 0.12)]
al_result = active_learn(vswr, bounds, al_cfg)

xs_al = al_result.history_x[:, 0]
ys_al = al_result.history_y
print(f"AL samples: {len(ys_al)}")
print(f"AL chosen x (mm): {sorted([round(x*1000, 1) for x in xs_al])}")

## Phase 2 — GP refit with ML hyperparameters

Fit a fresh GP on the AL data via `GaussianProcess.fit_ml`, which maximizes the log marginal likelihood over `(length_scale, sigma_f, sigma_n)`. Confirm the GP fits the surface well by plotting posterior mean / ±2σ vs the true function.

In [ ]:
x_train = np.array(xs_al).reshape(-1, 1)
y_train = np.array(ys_al)
gp = GaussianProcess.fit_ml(x_train, y_train,
                             initial_length_scale=0.02,
                             initial_sigma_f=1.0,
                             initial_sigma_n=1e-4)

x_dense = np.linspace(0.04, 0.12, 200)
y_true = np.array([vswr([x]) for x in x_dense])
y_pred = np.array([gp.predict_mean(np.array([x])) for x in x_dense])
y_var = np.array([gp.predict(np.array([x]))[1] for x in x_dense])
y_std = np.sqrt(np.maximum(y_var, 0.0))

rmse = float(np.sqrt(np.mean((y_pred - y_true) ** 2)))
print(f"GP RMSE on dense grid: {rmse:.4f}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(x_dense * 1000, y_true, "k-", label="true VSWR(L)", alpha=0.6)
ax.plot(x_dense * 1000, y_pred, "C0", label="GP posterior mean")
ax.fill_between(x_dense * 1000, y_pred - 2*y_std, y_pred + 2*y_std,
                alpha=0.2, color="C0", label="±2σ")
ax.scatter(xs_al * 1000, ys_al, c="C3", label="AL samples", zorder=5)
ax.axvline(LAMBDA_QUARTER * 1000, ls="--", color="k", label="λ/4")
ax.set_xlabel("L (mm)"); ax.set_ylabel("VSWR")
ax.legend(); ax.grid(True, alpha=0.3)
ax.set_title("After AL bootstrap + ML-fit GP")
plt.tight_layout(); plt.show()

## Phase 3 — BO with Expected Improvement

BO with EI, starting fresh. Same budget as a single-phase BO (no AL warm-start) would have used; compare convergence.

In [ ]:
bo_cfg = BoConfig(n_initial=5, n_iters=15, n_candidates=2048, xi=0.005, seed=7)
bo_result = bo_minimize(vswr, bounds, bo_cfg)
print(f"BO best length (mm): {bo_result.x_best[0]*1000:.3f}")
print(f"BO best VSWR:        {bo_result.y_best:.4f}")
print(f"BO total evals:      {len(bo_result.history_y)}")

How quickly did BO find the optimum? Plot best-so-far over evaluation count.

In [ ]:
ys = bo_result.history_y
best_so_far = np.minimum.accumulate(ys)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(np.arange(1, len(ys)+1), best_so_far, marker="o", color="C2",
        label="BO best so far")
ax.scatter(np.arange(1, len(ys)+1), ys, marker=".", alpha=0.4, color="C2",
           label="each evaluation")
ax.set_xlabel("BO evaluation #"); ax.set_ylabel("VSWR")
ax.legend(); ax.grid(True, alpha=0.3)
ax.set_title("BO convergence on the same synthetic objective")
plt.tight_layout(); plt.show()

## Next steps

Replace the closed-form `vswr` with a wrapper around a real `FdtdDriver` run. The AL+GP+BO loop is identical; only the cost-per-query changes.